<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/BinaryBaseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from datasets import load_dataset
import numpy as np

img_size = 150
batch_size = 50
num_classes = 2

In [ ]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 41780011-a6a9-47bb-a965-1586fe7872ec)')' thrown while requesting HEAD https://huggingface.co/datasets/Falah/Alzheimer_MRI/resolve/daac24f9597236b45837d82f7eb9c9ad1f8c60c8/.huggingface.yaml
Retrying in 1s [Retry 1/5].


data/train-00000-of-00001-c08a401c53fe53(…):   0%|          | 0.00/22.6M [00:00<?, ?B/s]

data/test-00000-of-00001-44110b9df98c558(…):   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1280 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5120
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1280
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented'])}


In [ ]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)

def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)
    new_channels = tf.shape(images)[-1]
    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_same)
    return images

def simple_preprocess(x):
    return x / 255.0

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.2),     # brightness adjustments
    tf.keras.layers.RandomZoom(0.1),           # zoom changes
    tf.keras.layers.RandomFlip("horizontal"),  # random horizontal flipping
])


# map original 4-class labels to binary:
#  label 0,1,3 (any demented) -> 1 (AD)
#  label 2 (Non_Demented)      -> 0 (Non-AD)
def map_to_binary_label(label_tensor):
    is_non_demented = tf.equal(label_tensor, 2)
    binary_label = tf.where(is_non_demented, 0, 1)
    return tf.cast(binary_label, tf.int32)

def to_tensorflow_dataset_binary(dataset_split, image_size, augment, preprocess_fn, shuffle=False):
    dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )

    def preprocess_input(example_dict):
        images = tf.cast(example_dict["image"], tf.float32)
        images = ensure_channel_dim(images)
        images = ensure_rgb_channels(images)
        images.set_shape([None, None, 3])
        images = tf.image.resize(images, (image_size, image_size))

        if augment:
            images = data_augmentation(images)

        images = preprocess_fn(images)

        original_label = tf.cast(example_dict["label"], tf.int32)
        binary_label = map_to_binary_label(original_label)

        return images, tf.cast(binary_label, tf.float32)

    dataset_tf = dataset_tf.map(
        preprocess_input,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset_tf = dataset_tf.batch(batch_size)
    return dataset_tf.prefetch(tf.data.AUTOTUNE)

# build binary datasets
train_ds = to_tensorflow_dataset_binary(train, img_size, True, simple_preprocess, shuffle=True)
val_ds   = to_tensorflow_dataset_binary(val,   img_size, False, simple_preprocess, shuffle=False)
test_ds  = to_tensorflow_dataset_binary(test,  img_size, False, simple_preprocess, shuffle=False)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),

    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(158, activation='relu'),
    layers.Dense(1, activation='sigmoid')   # binary output
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 158)            │     6,552,734 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           159 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,557,981 (25.02 MB)

 Trainable params: 6,557,981 (25.02 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
opt = optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[reduce_lr, early_stop]
)

test_loss, test_acc = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 19s 152ms/step - accuracy: 0.5424 - loss: 0.8043 - val_accuracy: 0.7119 - val_loss: 0.5676 - learning_rate: 0.0010
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.6847 - loss: 0.5872 - val_accuracy: 0.7285 - val_loss: 0.5338 - learning_rate: 0.0010
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 121ms/step - accuracy: 0.7088 - loss: 0.5602 - val_accuracy: 0.7529 - val_loss: 0.4999 - learning_rate: 0.0010
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 121ms/step - accuracy: 0.7462 - loss: 0.5142 - val_accuracy: 0.7627 - val_loss: 0.4990 - learning_rate: 0.0010
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step - accuracy: 0.7630 - loss: 0.4884 - val_accuracy: 0.7373 - val_loss: 0.5491 - learning_rate: 0.0010
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step - accuracy: 0.7845 - loss: 0.4582 - val_accuracy: 0.7900 - val_loss: 0.4375 - learning_rate: 0.0010
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step - accuracy: 0.7982 - l

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend((preds.flatten() >= 0.5).astype(int))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))
print("Model test accuracy: ", accuracy_score(y_true, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
[[615  19]
 